_We'll build this together, live, using everything from this session: nested for loops and string slicing._

## Application: Disney+ Account Sharing

Let's bring together everything from this session (nested for loops and string slicing) on a real example.

<img src="images/netflix.png" alt="News article about the streaming sharing crackdown" width="500">

Disney estimates that **50 million households** are sharing Disney+ accounts outside of the household ([source](https://tinyurl.com/forbes-disney)). Following in Netflix's footsteps, Disney is now tracking the geographic locations where each device streams content.

An account can be used by several **devices**, ideally all belonging to one household. Two devices used by the *same* person tend to move together, so their location histories share long stretches in common. When *different* people share one account, the devices are used more independently, and their histories overlap much less.

Each location is logged as a 3-letter city code (e.g. `lon`, `fra`, `dub`), joined with underscores. Given the last 20 locations for two devices on one account, we want to find the **longest sequence of cities that both devices visited, in the same order**.

A long shared sequence points to normal, single-household use. A short one is a *hint*, not proof on its own, that different people may be sharing the account.

## Part 1: A naive solution

Let's build this step by step.

**Step 1: generate every contiguous chunk of `cities1`.** We need two nested loops, one for where a chunk starts and one for where it ends, so we can try every possible slice.

In [ ]:
def find_shared_pattern(cities1, cities2):
    """Find the longest sequence of locations shared between two devices."""
    # your code here



**Step 2: slice out each chunk.** `cities1[start:end]` gives us the candidate sequence to test.

In [ ]:
def find_shared_pattern(cities1, cities2):
    """Find the longest sequence of locations shared between two devices."""
    # your code here



**Step 3: check membership and keep the longest match.** If `seq` also appears in `cities2`, and it's longer than the best one found so far, it becomes our new best.

In [ ]:
def find_shared_pattern(cities1, cities2):
    """Find the longest sequence of locations shared between two devices."""
    # your code here



In [ ]:
# Last 20 locations accessed from each device
device1 = "lon_fra_dub_lon_fra_ber_lon_fra_dub_lon_fra_mad_mil_vie_lon_fra_dub_lon_fra_ber"
device2 = "mil_rom_lon_fra_dub_lon_fra_dub_lon_fra_dub_lon_fra_dub_lon_fra_mad_mil_ath_mil"

**Now test it on the real data:**

In [ ]:
shared = find_shared_pattern(device1, device2)
print(f"Found this shared sequence: {shared}")

shared_cities = shared.split("_")
print(f"As a list of cities: {shared_cities}")
print(f"Number of shared visits: {len(shared_cities)}")

### How expensive is this?

`find_shared_pattern` works, but it does a lot of repeated work. Before improving it, analyse its time complexity.

Both strings have the same length here (20 locations each), so let `n` be that length.

- How many `(start, end)` pairs do the two nested loops try?
- For each pair, roughly what does building the slice `cities1[start:end]` cost? And the `seq in cities2` check?
- Put these together: what is the overall time complexity, in terms of `n`?

## Part 2: Using the city structure

Before jumping to something asymptotically faster, notice `find_shared_pattern` works one **character** at a time, even though the real unit here is a **city** (always 3 letters, separated by `_`). Let's reuse the same nested-loop idea, one level up: chunks of *cities* instead of chunks of *characters*.

**Step 1: turn the string into chunks of whole cities.** Split `cities1` into a list of city codes with `.split("_")`. Then, for every `(start, end)` pair, slice the list and join it back together with `"_".join(...)` to get a chunk in the same format as `cities2`.

In [ ]:
def find_shared_pattern_by_city(cities1, cities2):
    """Same idea as find_shared_pattern, but working with whole cities instead of characters."""
    # your code here



**Step 2: check membership and keep the longest match.** Same idea as before, just applied to `chunk` instead of a character slice.

In [ ]:
def find_shared_pattern_by_city(cities1, cities2):
    """Same idea as find_shared_pattern, but working with whole cities instead of characters."""
    # your code here



**Now test it:**

In [ ]:
shared = find_shared_pattern_by_city(device1, device2)
print(f"Found this shared sequence: {shared}")

shared_cities = shared.split("_")
print(f"As a list of cities: {shared_cities}")
print(f"Number of shared visits: {len(shared_cities)}")

**Why might this be faster in practice?**

- What does `n` refer to in each version: characters, or cities?
- How do those two sizes compare, for these particular device histories?
- Does the complexity *class* change, or just the size of `n`?

## Part 3: A faster approach

The slow part of both solutions above is re-searching `cities2` from scratch for every candidate chunk. Here's a different way to think about the problem entirely.

**The big idea:** line the two devices' full histories up against each other, shifted left or right by some amount, and see how much they agree while lined up that way. Slide one across the other, one position at a time; whichever position produces the longest run of agreement is the longest shared sequence. So the problem becomes: try every possible shift, and for each one, find the longest run of matching cities.

**Line the two strings up at an offset.** Imagine sliding `cities1` along `cities2`; wherever they overlap, character `i` of `cities1` sits directly above character `j = i + offset` of `cities2`. For example, with `cities1 = "abcde"` and `cities2 = "xxabc"` and `offset = 2`:

```
      j:  0 1 2 3 4
cities2:  x x a b c
cities1:      a b c d e
      i:      0 1 2 3 4
```

`cities1[0]` lines up with `cities2[2]`, `cities1[1]` with `cities2[3]`, and so on. Reading down the overlapping columns, `a b c` matches: a run of length 3.

**Scan one offset in a single pass.** Walk `i` from `0` to `len(cities1) - 1`, keeping a `run` string:

- if `j = i + offset` is a valid index of `cities2` *and* `cities1[i] == cities2[j]`, extend `run` by that character;
- otherwise, reset `run` to `""`;
- after each step, if `run` is longer than the best run seen so far, remember it.

**Try every offset.** The alignments range from `cities1` shifted fully left (`offset = -(len(cities1) - 1)`, its last character over `cities2[0]`) to fully right (`offset = len(cities2) - 1`, its first character over the last character of `cities2`). The overall answer is the longest `run` found across all of them.

There are only about `2n` offsets, and each one scans the strings just once: far less repeated work than testing every chunk separately. Work out its complexity too.

Implement it below. It should return the same result as `find_shared_pattern`.

In [ ]:
def find_shared_pattern_fast(cities1, cities2):
    """Same result as find_shared_pattern, but far fewer operations."""
    # your code here



### Beyond this module

Even the offset method still repeats some work. There is a genuinely clever algorithm that finds the longest common substring in **linear time**, `O(m + n)`, using an advanced data structure called a *suffix tree*. That is well beyond this module, but it is a nice example of how choosing the right data structure can collapse a problem's cost.